
# Decision Tree Baseline — Kaggle Submission

This notebook:
- Preprocesses numeric/categorical features with a sklearn Pipeline
- Evaluates a DecisionTreeRegressor with 5-fold GroupKFold (grouped by `household_id`)
- Trains on full train and generates `submission_dt.csv`


## 1) Imports & Config

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Paths
BASE = Path(".")
TRAIN_PATH = BASE / "train/train.csv"
TEST_PATH  = BASE / "test/test.csv"
SUB_PATH   = "/kaggle/working/submission_dt.csv"

TARGET = "monthly_consumption_kwh"
ROW_ID = "row_id"
GROUP_COL = "household_id"   # used for GroupKFold

RANDOM_STATE = 42
N_SPLITS = 5


## 2) Load Data

In [2]:

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError("Expected train/test at ./kaggle_split/. Please generate them first.")

train = pd.read_csv(TRAIN_PATH, low_memory=False)
test  = pd.read_csv(TEST_PATH, low_memory=False)

print("Train shape:", train.shape, "| Test shape:", test.shape)
display(train.head(3))
display(test.head(3))


Train shape: (120000, 71) | Test shape: (120000, 71)


,household_id,state,location,climate_region,dwelling_type,floor_number,building_age,floor_area_sqft,airconditioned_area_pct,num_rooms,...,rainfall_max_mm_day,rainfall_mean_mm_day,rainfall_total_mm,solar_irradiance_min_wm2,solar_irradiance_max_wm2,solar_irradiance_mean_wm2,wind_speed_min_mps,wind_speed_max_mps,wind_speed_mean_mps,monthly_consumption_kwh
0,H00001,RJ,Metropolitan,arid,Apartment,6.0,26,1010.0,40,3,...,20.4,0.8,24.7,215.8,293.4,259.7,1.5,4.6,3.1,251.66
1,H00001,RJ,Metropolitan,arid,Apartment,6.0,26,1010.0,40,3,...,24.1,2.9,89.2,98.0,177.6,138.3,1.0,3.4,2.1,781.68
2,H00001,RJ,Metropolitan,arid,Apartment,6.0,26,1010.0,40,3,...,25.7,2.7,83.2,227.7,307.7,258.4,1.9,4.4,3.1,221.32


,row_id,household_id,state,location,climate_region,dwelling_type,floor_number,building_age,floor_area_sqft,airconditioned_area_pct,...,rainfall_min_mm_day,rainfall_max_mm_day,rainfall_mean_mm_day,rainfall_total_mm,solar_irradiance_min_wm2,solar_irradiance_max_wm2,solar_irradiance_mean_wm2,wind_speed_min_mps,wind_speed_max_mps,wind_speed_mean_mps
0,0,H10001,KA,Village,inland,Apartment,2.0,38,890.0,35,...,0.0,28.3,2.9,86.5,196.8,281.8,242.4,2.5,4.8,3.5
1,1,H10001,KA,Village,inland,Apartment,2.0,38,890.0,35,...,0.0,18.6,7.0,218.3,87.3,168.9,129.9,1.1,3.6,2.4
2,2,H10001,KA,Village,inland,Apartment,2.0,38,890.0,35,...,0.0,15.2,1.2,36.2,171.9,264.0,222.9,1.4,3.9,2.6



## 3) Features & Basic Preprocessing

- Keep **all** columns except target as features.
- For `test.csv`, `row_id` is present and must be excluded from learning but used later for submission.
- We’ll include `household_id` as a **categorical feature**.


In [3]:

# Separate target and groups
y = train[TARGET].astype(float)
groups = train[GROUP_COL].astype(str)

# Build feature sets (exclude target only)
X = train.drop(columns=[TARGET])
X_test = test.copy()

# Remember row ids for the submission
row_id_test = X_test[ROW_ID].values

# Identify numeric and categorical cols
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

# row_id is not a feature; remove from both X and X_test if present
if ROW_ID in num_cols:  num_cols.remove(ROW_ID)
if ROW_ID in cat_cols:  cat_cols.remove(ROW_ID)

print("Numeric features:", len(num_cols))
print("Categorical features:", len(cat_cols))


Numeric features: 57
Categorical features: 13



## 4) Build Pipeline

- Numeric: median imputation  
- Categorical: most-frequent imputation + OneHotEncoder  
- Model: DecisionTreeRegressor

*(Using `sparse=True` for older sklearn compatibility.)*


In [4]:

numeric_pipe = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
])

categorical_pipe = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=True)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

model = DecisionTreeRegressor(
    random_state=RANDOM_STATE,
    max_depth=None,         # try setting e.g. 12 for more bias / less variance
    min_samples_leaf=2
)

pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model),
])


## 5) Cross-Validation (GroupKFold by household)

In [5]:

# gkf = GroupKFold(n_splits=N_SPLITS)

# fold_metrics = []
# for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups=groups), 1):
#     X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
#     y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    
#     pipe.fit(X_tr, y_tr)
#     pred = pipe.predict(X_va)
    
#     mae  = mean_absolute_error(y_va, pred)
#     rmse = mean_squared_error(y_va, pred, squared=False)
#     r2   = r2_score(y_va, pred)
#     fold_metrics.append((mae, rmse, r2))
#     print(f"Fold {fold}: MAE={mae:,.3f} | RMSE={rmse:,.3f} | R2={r2:,.4f}")

# mae_mean  = np.mean([m[0] for m in fold_metrics])
# rmse_mean = np.mean([m[1] for m in fold_metrics])
# r2_mean   = np.mean([m[2] for m in fold_metrics])
# print("\nCV Mean: MAE={:,.3f} | RMSE={:,.3f} | R2={:.4f}".format(mae_mean, rmse_mean, r2_mean))


## 6) Train on Full Train & Predict Test

In [6]:

pipe.fit(X, y)
test_pred = pipe.predict(X_test)

# Safety: ensure non-negative predictions if your target cannot be negative
test_pred = np.clip(test_pred, 0, None)

submission = pd.DataFrame({
    "row_id": row_id_test.astype(np.int64),
    "monthly_consumption_kwh": test_pred.astype(float)
}).sort_values("row_id").reset_index(drop=True)

submission.to_csv(SUB_PATH, index=False)
print("Wrote submission to:", SUB_PATH)
display(submission.head(10))


Wrote submission to: /kaggle/working/submission_dt.csv


,row_id,monthly_consumption_kwh
0,0,220.883333
1,1,468.030000
2,2,183.710000
3,3,167.440000
4,4,177.806667
5,5,339.080000
6,6,422.480000
7,7,224.195000
8,8,231.346667
9,9,236.895000


## 7) (Optional) Save Model Artifact

In [7]:

# import joblib
# joblib.dump(pipe, "decision_tree_pipeline.joblib")
# print("Saved model to decision_tree_pipeline.joblib")


In [8]:
test

,row_id,household_id,state,location,climate_region,dwelling_type,floor_number,building_age,floor_area_sqft,airconditioned_area_pct,...,rainfall_min_mm_day,rainfall_max_mm_day,rainfall_mean_mm_day,rainfall_total_mm,solar_irradiance_min_wm2,solar_irradiance_max_wm2,solar_irradiance_mean_wm2,wind_speed_min_mps,wind_speed_max_mps,wind_speed_mean_mps
0,0,H10001,KA,Village,inland,Apartment,2.0,38,890.0,35,...,0.0,28.3,2.9,86.5,196.8,281.8,242.4,2.5,4.8,3.5
1,1,H10001,KA,Village,inland,Apartment,2.0,38,890.0,35,...,0.0,18.6,7.0,218.3,87.3,168.9,129.9,1.1,3.6,2.4
2,2,H10001,KA,Village,inland,Apartment,2.0,38,890.0,35,...,0.0,15.2,1.2,36.2,171.9,264.0,222.9,1.4,3.9,2.6
3,3,H10001,KA,Village,inland,Apartment,2.0,38,890.0,35,...,0.0,37.8,3.4,98.5,219.7,287.0,250.6,1.8,4.3,2.8
4,4,H10001,KA,Village,inland,Apartment,2.0,38,890.0,35,...,0.0,28.1,2.2,67.5,191.7,284.3,242.5,0.2,2.7,1.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119995,119995,H20000,TN,Village,inland,Villa,0.0,12,1010.0,10,...,0.0,42.4,3.0,94.0,212.6,283.8,254.4,1.2,3.9,2.3
119996,119996,H20000,TN,Village,inland,Villa,0.0,12,1010.0,10,...,0.0,73.5,9.0,277.8,199.8,265.9,229.2,1.1,4.5,2.9
119997,119997,H20000,TN,Village,inland,Villa,0.0,12,1010.0,10,...,0.0,19.5,2.9,86.2,161.3,264.7,206.5,0.8,2.8,1.7
119998,119998,H20000,TN,Village,inland,Villa,0.0,12,1010.0,10,...,0.0,21.8,3.1,96.4,151.1,216.9,195.5,0.7,3.9,2.3
